# Day 27 - Edit distance and similarity: Levenshtein, Jaccard, MinHash

Days 24-26 answered exact questions: does this pattern occur, where, what is
repeated verbatim. Real text is messier - a typo, a dropped word, a cookie
banner - and an exact structure reports all of those as *no match*.

Two families of answer, and the gap between them is the point of today:

* **Edit distance** measures how far apart two strings are, exactly, by dynamic
  programming. O(n*m) per pair: fine for one spelling correction, hopeless for
  a corpus.
* **Jaccard + MinHash + LSH** trade exactness for scale: a fixed-size signature
  estimates set overlap, and banding turns "compare all pairs" into "look in
  the same bucket".

Standard library only.

## 1. The Levenshtein table

`d[i][j]` is the distance between `a[:i]` and `b[:j]`. Exactly three
predecessors feed each cell, and each one is an operation:

```
d[i-1][j]   + 1          delete a[i-1]
d[i][j-1]   + 1          insert b[j-1]
d[i-1][j-1] + (a != b)   substitute, or match for free
```

The first row and column are the base cases: turning a prefix into the empty
string costs one deletion per character.

In [1]:
def edit_table(a, b):
    n, m = len(a), len(b)
    d = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        d[i][0] = i
    for j in range(m + 1):
        d[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            d[i][j] = min(d[i - 1][j] + 1,          # delete
                          d[i][j - 1] + 1,          # insert
                          d[i - 1][j - 1] + cost)   # substitute / match
    return d


def edit_distance(a, b):
    return edit_table(a, b)[len(a)][len(b)]


A, B = 'kitten', 'sitting'
T = edit_table(A, B)
print('     ' + '  '.join('%2s' % c for c in ' ' + B))
for i, row in enumerate(T):
    print('  %s  ' % (' ' if i == 0 else A[i - 1])
          + '  '.join('%2d' % v for v in row))
print('\ndistance =', edit_distance(A, B))

          s   i   t   t   i   n   g
      0   1   2   3   4   5   6   7
  k   1   1   2   3   4   5   6   7
  i   2   2   1   2   3   4   5   6
  t   3   3   2   1   2   3   4   5
  t   4   4   3   2   1   2   3   4
  e   5   5   4   3   2   2   3   4
  n   6   6   5   4   3   3   2   3

distance = 3


## 2. Backtracking: the distance is rarely what you actually want

A spell checker wants the correction, a diff wants the hunks, an ASR report
wants to know how many words were *substituted* rather than *dropped*. All of
that is free: walk backwards from the corner and ask which predecessor each
value came from.

In [2]:
def edit_ops(a, b):
    d = edit_table(a, b)
    i, j, ops = len(a), len(b), []
    while i > 0 or j > 0:
        if i > 0 and j > 0:
            cost = 0 if a[i - 1] == b[j - 1] else 1
            if d[i][j] == d[i - 1][j - 1] + cost:
                ops.append(('match' if cost == 0 else 'substitute',
                            i - 1, a[i - 1], b[j - 1]))
                i, j = i - 1, j - 1
                continue
        if i > 0 and d[i][j] == d[i - 1][j] + 1:
            ops.append(('delete', i - 1, a[i - 1], ''))
            i -= 1
            continue
        ops.append(('insert', i, '', b[j - 1]))
        j -= 1
    ops.reverse()
    return ops


for op, pos, frm, to in edit_ops(A, B):
    if op != 'match':
        print('%-10s at %d: %s -> %s' % (op, pos, frm or '-', to or '-'))

substitute at 0: k -> s
substitute at 4: e -> i
insert     at 6: - -> g


## 3. One row is enough - unless you want the script back

Every cell reads `d[i-1][j]`, `d[i][j-1]` and `d[i-1][j-1]`, all of which live
in the current row or the one above it. Keeping the whole table is only
necessary for the backtrace.

In [3]:
def edit_distance_rows(a, b):
    if len(a) < len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i] + [0] * len(b)
        for j, cb in enumerate(b, 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1,
                         prev[j - 1] + (ca != cb))
        prev = cur
    return prev[len(b)]


def similarity_ratio(a, b):
    if not a and not b:
        return 1.0
    return 1.0 - edit_distance(a, b) / max(len(a), len(b))


print(edit_distance_rows(A, B), edit_distance(A, B))
print('similarity %.3f' % similarity_ratio(A, B))

3 3
similarity 0.571


## 4. The bounded version - the only one that is fast enough in practice

Almost every real caller has a threshold: "did the user mistype this by at most
2", "is this a near-duplicate". Under a threshold `k`, only a diagonal band of
width `2k+1` can matter, because leaving the diagonal by one cell already costs
one operation. That makes the cost `O(k * min(n, m))`, and the length
difference alone answers some queries with no table at all.

In [4]:
def edit_distance_bounded(a, b, k):
    n, m = len(a), len(b)
    if abs(n - m) > k:              # every extra character costs an indel
        return k + 1, 0
    INF = k + 1
    prev = [INF] * (m + 1)
    for j in range(min(k, m) + 1):
        prev[j] = j
    cells = 0
    for i in range(1, n + 1):
        cur = [INF] * (m + 1)
        lo, hi = max(1, i - k), min(m, i + k)
        if lo == 1:
            cur[0] = i
        for j in range(lo, hi + 1):
            cells += 1
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1,
                         prev[j - 1] + (a[i - 1] != b[j - 1]))
        if min(cur[lo:hi + 1] or [INF]) > k:      # whole band exceeded k
            return k + 1, cells
        prev = cur
    return (prev[m], cells) if prev[m] <= k else (k + 1, cells)


for k in (1, 2, 3):
    d, cells = edit_distance_bounded(A, B, k)
    print('k = %d: %-6s cells %2d  (full table %d)'
          % (k, d if d <= k else '> %d' % k, cells, len(A) * len(B)))

LA = 'the quick brown fox jumps over the lazy dog ' * 3
LB = LA.replace('quick', 'quack')
d3, c3 = edit_distance_bounded(LA, LB, 3)
print('\n%d chars, k = 3: distance %d, %d cells vs %d (%.1f%%)'
      % (len(LA), d3, c3, len(LA) * len(LB),
         100.0 * c3 / (len(LA) * len(LB))))
print('k = 1: stopped after %d cells' % edit_distance_bounded(LA, LB, 1)[1])

k = 1: > 1    cells 14  (full table 42)
k = 2: > 2    cells 26  (full table 42)
k = 3: 3      cells 33  (full table 42)

132 chars, k = 3: distance 3, 912 cells vs 17424 (5.2%)
k = 1: stopped after 152 cells


## 5. Same table, a different alphabet: word error rate

WER is the standard ASR metric, and it is edit distance where the "characters"
are whole words. The breakdown matters more than the number: substitutions,
deletions and insertions have very different causes in a speech system.

In [5]:
def edit_ops_seq(a, b):
    n, m = len(a), len(b)
    d = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        d[i][0] = i
    for j in range(m + 1):
        d[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            d[i][j] = min(d[i - 1][j] + 1, d[i][j - 1] + 1,
                          d[i - 1][j - 1] + (a[i - 1] != b[j - 1]))
    i, j, ops = n, m, []
    while i > 0 or j > 0:
        if i > 0 and j > 0:
            cost = 0 if a[i - 1] == b[j - 1] else 1
            if d[i][j] == d[i - 1][j - 1] + cost:
                ops.append(('match' if cost == 0 else 'substitute',
                            a[i - 1], b[j - 1]))
                i, j = i - 1, j - 1
                continue
        if i > 0 and d[i][j] == d[i - 1][j] + 1:
            ops.append(('delete', a[i - 1], None)); i -= 1
            continue
        ops.append(('insert', None, b[j - 1])); j -= 1
    ops.reverse()
    return ops


def word_error_rate(reference, hypothesis):
    ref, hyp = reference.split(), hypothesis.split()
    ops = edit_ops_seq(ref, hyp)
    c = {'substitute': 0, 'delete': 0, 'insert': 0, 'match': 0}
    for op, _, _ in ops:
        c[op] += 1
    errors = c['substitute'] + c['delete'] + c['insert']
    return errors / max(len(ref), 1), c, ops


ref = 'the model streams tokens back to the client'
hyp = 'the model streamed tokens to a client'
wer, c, ops = word_error_rate(ref, hyp)
print('WER = (%dS + %dD + %dI) / %d = %.3f'
      % (c['substitute'], c['delete'], c['insert'], len(ref.split()), wer))
for op, frm, to in ops:
    if op != 'match':
        print('  %-10s %-10s -> %s' % (op, frm or '-', to or '-'))

WER = (2S + 1D + 0I) / 8 = 0.375
  substitute streams    -> streamed
  delete     back       -> -
  substitute the        -> a


## 6. Delete substitution from the menu and you get diff

A line cannot be *edited*, only added or removed, so the cheapest script is the
one that keeps the most common lines - that is the longest common subsequence,
and it is what `diff` computes.

In [6]:
def lcs_length(a, b):
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b, 1):
            cur.append(prev[j - 1] + 1 if x == y else max(prev[j], cur[j - 1]))
        prev = cur
    return prev[len(b)]


def diff_script(a, b):
    n, m = len(a), len(b)
    d = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            d[i][j] = (d[i - 1][j - 1] + 1 if a[i - 1] == b[j - 1]
                       else max(d[i - 1][j], d[i][j - 1]))
    i, j, out = n, m, []
    while i > 0 and j > 0:
        if a[i - 1] == b[j - 1]:
            out.append(('keep', a[i - 1])); i, j = i - 1, j - 1
        elif d[i - 1][j] >= d[i][j - 1]:
            out.append(('delete', a[i - 1])); i -= 1
        else:
            out.append(('insert', b[j - 1])); j -= 1
    while i > 0:
        out.append(('delete', a[i - 1])); i -= 1
    while j > 0:
        out.append(('insert', b[j - 1])); j -= 1
    out.reverse()
    return out


old = ['import os', 'def main():', '    run()', '    return 0']
new = ['import os', 'import sys', 'def main():', '    return 0']
print('LCS length =', lcs_length(old, new))
for op, line in diff_script(old, new):
    print(' %s %s' % ({'keep': ' ', 'delete': '-', 'insert': '+'}[op], line))

LCS length = 3
   import os
 + import sys
   def main():
 -     run()
       return 0


## 7. Documents: shingles and Jaccard

Edit distance on two 5 kB documents is 25 million cells. Shingling throws away
the ordering above the k-gram level and keeps a *set*, and sets have a
similarity measure that costs nothing.

`k = 5` words is the usual choice: short enough that real copies share thousands
of shingles, long enough that unrelated English shares almost none.

In [7]:
def shingles(text, k=5):
    words = text.lower().split()
    if len(words) < k:
        return {' '.join(words)} if words else set()
    return {' '.join(words[i:i + k]) for i in range(len(words) - k + 1)}


def jaccard(a, b):
    if not a and not b:
        return 1.0
    return len(a & b) / len(a | b)


DOC_A = ('machine learning models are trained on large corpora of text '
         'collected from the public web and the same paragraph often appears '
         'in thousands of pages so a training set that is not deduplicated '
         'will show the model the same sentences again and again')
DOC_B = DOC_A.replace('public web', 'open web')
DOC_C = DOC_A + ' copyright notices and cookie banners make this worse'
DOC_D = ('a suffix array sorts every suffix of a string so that any pattern '
         'can be located with two binary searches over the sorted order')

sa, sb, sc, sd = (shingles(DOC_A), shingles(DOC_B),
                  shingles(DOC_C), shingles(DOC_D))
print('|A| = %d shingles' % len(sa))
print('J(A, B) = %.3f   one phrase swapped' % jaccard(sa, sb))
print('J(A, C) = %.3f   a clause appended' % jaccard(sa, sc))
print('J(A, D) = %.3f   different topic' % jaccard(sa, sd))

|A| = 39 shingles
J(A, B) = 0.773   one phrase swapped
J(A, C) = 0.830   a clause appended
J(A, D) = 0.000   different topic


## 8. MinHash: the collision probability *is* the Jaccard

Under a random permutation of the universe, the odds that two sets share a
minimum element are the odds that the minimum of their *union* happens to lie
in their *intersection* - which is exactly `|A n B| / |A u B|`.

So each hash function is one Bernoulli trial with success probability J, and
the fraction of matching slots is an unbiased estimate. Note the hash: Python's
built-in `hash()` on `str` is randomised per process, so a signature built with
it would differ between runs.

In [8]:
import hashlib


def _hash(item, seed):
    h = hashlib.blake2b(item.encode('utf-8'), digest_size=8,
                        key=seed.to_bytes(8, 'little'))
    return int.from_bytes(h.digest(), 'little')


def minhash_signature(s, num_hashes=128):
    if not s:
        return [0] * num_hashes
    return [min(_hash(x, seed) for x in s) for seed in range(num_hashes)]


def estimate_jaccard(sig_a, sig_b):
    return sum(1 for x, y in zip(sig_a, sig_b) if x == y) / len(sig_a)


truth = jaccard(sa, sb)
print('true J = %.3f' % truth)
for K in (16, 64, 256, 1024):
    est = estimate_jaccard(minhash_signature(sa, K), minhash_signature(sb, K))
    print('  K = %4d: %.3f   error %.3f   predicted +-%.3f'
          % (K, est, abs(est - truth), (truth * (1 - truth) / K) ** 0.5))

true J = 0.773
  K =   16: 0.688   error 0.085   predicted +-0.105
  K =   64: 0.828   error 0.055   predicted +-0.052
  K =  256: 0.766   error 0.007   predicted +-0.026
  K = 1024: 0.771   error 0.002   predicted +-0.013


## 9. LSH banding: stop enumerating pairs at all

Split each signature into `b` bands of `r` rows and hash each band. Two
documents become candidates if *any one* band matches exactly. A single band of
`r` rows matches with probability `J^r`, so

```
P(candidate) = 1 - (1 - J^r)^b
```

which is an S-curve in J, steep around `(1/b)^(1/r)`. Choosing `b` and `r` *is*
choosing the threshold. LSH is a filter, not an answer: the surviving pairs are
still verified exactly.

In [9]:
def band_buckets(signatures, bands, rows):
    buckets = {}
    for idx, sig in enumerate(signatures):
        for bi in range(bands):
            strip = tuple(sig[bi * rows:(bi + 1) * rows])
            buckets.setdefault((bi, strip), []).append(idx)
    return buckets


def candidate_pairs(signatures, bands, rows):
    pairs = set()
    for members in band_buckets(signatures, bands, rows).values():
        if len(members) > 1:
            for x in range(len(members)):
                for y in range(x + 1, len(members)):
                    pairs.add((members[x], members[y]))
    return pairs


def lsh_threshold(bands, rows):
    return (1.0 / bands) ** (1.0 / rows)


def prob_candidate(j, bands, rows):
    return 1.0 - (1.0 - j ** rows) ** bands


for bands, rows in ((32, 4), (16, 8), (8, 16)):
    print('b = %2d, r = %2d: threshold ~ %.2f   P(J=0.3) = %.3f   '
          'P(J=0.8) = %.3f'
          % (bands, rows, lsh_threshold(bands, rows),
             prob_candidate(0.3, bands, rows),
             prob_candidate(0.8, bands, rows)))

b = 32, r =  4: threshold ~ 0.42   P(J=0.3) = 0.229   P(J=0.8) = 1.000
b = 16, r =  8: threshold ~ 0.71   P(J=0.3) = 0.001   P(J=0.8) = 0.947
b =  8, r = 16: threshold ~ 0.88   P(J=0.3) = 0.000   P(J=0.8) = 0.204


In [10]:
DOCS = [DOC_A, DOC_B, DOC_C, DOC_D,
        DOC_D.replace('every suffix', 'all suffixes'),
        ('the scheduler admits requests into the running batch until the '
         'batch is full and then runs one forward pass over the whole batch '
         'so that the weights are read from memory once instead of once per '
         'request which is where the throughput comes from'),
        ('bananas grow in tropical climates and are harvested green before '
         'they are shipped in refrigerated containers and then ripened in a '
         'controlled room with ethylene gas a few days before they reach the '
         'shelves of a supermarket')]

sets = [shingles(d) for d in DOCS]
sigs = [minhash_signature(s, 128) for s in sets]
cands = candidate_pairs(sigs, 32, 4)

exact = [(i, j) for i in range(len(DOCS)) for j in range(i + 1, len(DOCS))
         if jaccard(sets[i], sets[j]) >= 0.5]
n_pairs = len(DOCS) * (len(DOCS) - 1) // 2
print('%d documents -> %d pairs, LSH proposed %d candidates (%.0f%%)'
      % (len(DOCS), n_pairs, len(cands), 100.0 * len(cands) / n_pairs))
verified = sorted((i, j) for i, j in cands
                  if jaccard(sets[i], sets[j]) >= 0.5)
for i, j in verified:
    print('  doc%d ~ doc%d   J = %.3f' % (i, j, jaccard(sets[i], sets[j])))
print('same answer as brute force:', verified == sorted(exact))

7 documents -> 21 pairs, LSH proposed 4 candidates (19%)
  doc0 ~ doc1   J = 0.773
  doc0 ~ doc2   J = 0.830
  doc1 ~ doc2   J = 0.654
  doc3 ~ doc4   J = 0.538
same answer as brute force: True


## 10. LeetCode 72, 1143 and 161

`161` is the special case that motivates the band: at `k = 1` the table
collapses to a single scan.

In [11]:
def min_distance(word1, word2):
    """LC 72 - Edit Distance."""
    return edit_distance_rows(word1, word2)


def longest_common_subsequence(text1, text2):
    """LC 1143 - the same table without substitution."""
    return lcs_length(text1, text2)


def is_one_edit_distance(s, t):
    """LC 161 - distance exactly 1, in O(n) time and O(1) space."""
    if len(s) > len(t):
        s, t = t, s
    if len(t) - len(s) > 1 or s == t:
        return False
    for i in range(len(s)):
        if s[i] != t[i]:
            if len(s) == len(t):
                return s[i + 1:] == t[i + 1:]     # substitute
            return s[i:] == t[i + 1:]             # insert into s
    return len(t) == len(s) + 1


print(min_distance('horse', 'ros'), min_distance('intention', 'execution'))
print(longest_common_subsequence('abcde', 'ace'))
print(is_one_edit_distance('ab', 'acb'), is_one_edit_distance('cab', 'ad'))

3 5
3
True False


## 11. Tests

If any of these fail, something above is wrong.

In [12]:
assert edit_distance('kitten', 'sitting') == 3
assert edit_distance_rows('kitten', 'sitting') == 3
assert edit_distance('', 'abc') == 3 and edit_distance('abc', 'abc') == 0
ops = edit_ops('kitten', 'sitting')
assert sum(1 for o in ops if o[0] != 'match') == 3
assert ''.join(to for op, _, _, to in ops if op != 'delete') == 'sitting'
assert edit_distance_bounded('kitten', 'sitting', 1)[0] == 2      # k + 1
assert edit_distance_bounded('kitten', 'sitting', 3)[0] == 3
assert edit_distance_bounded('abc', 'abcdef', 2) == (3, 0)        # length gate
assert lcs_length('abcde', 'ace') == 3 and lcs_length('abc', 'def') == 0
assert word_error_rate('a b c d', 'a b c d')[0] == 0.0
assert jaccard(sa, sa) == 1.0 and jaccard(sa, set()) == 0.0
assert minhash_signature(sa, 32) == minhash_signature(sa, 32)     # reproducible
assert estimate_jaccard(minhash_signature(sa, 64),
                        minhash_signature(sa, 64)) == 1.0
assert abs(estimate_jaccard(minhash_signature(sa, 512),
                            minhash_signature(sb, 512)) - truth) < 0.08
assert len(cands) < n_pairs and verified == sorted(exact)
assert min_distance('horse', 'ros') == 3
assert is_one_edit_distance('ab', 'acb') and not is_one_edit_distance('ab', 'ab')
print('all assertions passed')

all assertions passed
